# Project: A CNN that *is* the Game of Life 🧬
Conway's Game of Life is a convolution in disguise: each cell looks at its 3×3 neighbourhood. You'll (1) implement the rules, (2) **set the weights of a tiny CNN by hand** so it computes Life *exactly*, and (3) try to *train* the same network from random weights. Surprise: with the minimal width, training doesn't always succeed.

Topic: [[CNN]], [[Neural Networks]] · guide note: [[Project - Game of Life CNN]] · paper: [Springer & Kenyon 2020, *It's Hard for Neural Networks To Learn the Game of Life*](https://arxiv.org/abs/2009.01398)

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
torch.set_num_threads(4)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

## 1. The rules (on a torus)
A live cell with 2 or 3 live neighbours survives; a dead cell with exactly 3 becomes alive; everything else dies. Edges wrap around (`np.roll`). Return the next board (same dtype).

In [ ]:
def life_step(board):
    # TODO 1: count the 8 neighbours with np.roll, then apply the rules
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_life():
    blinker = np.zeros((5, 5), int); blinker[2, 1:4] = 1
    glider = np.zeros((8, 8), int); glider[[0, 1, 2, 2, 2], [1, 2, 0, 1, 2]] = 1
    g4 = glider
    for _ in range(4): g4 = life_step(g4)
    return np.array_equal(life_step(blinker), blinker.T) and np.array_equal(g4, np.roll(glider, (1, 1), (0, 1)))
check("life rules", _t_life, "A blinker must flip to vertical; a glider must move 1 down-right every 4 steps.")
if ready("life rules"):
    r = np.random.default_rng(0); B = (r.random((40, 40)) < 0.3).astype(int)
    fig, ax = plt.subplots(1, 4, figsize=(12, 3))
    for a, t in zip(ax, range(4)):
        a.imshow(B, cmap="binary"); a.set_title(f"t = {t * 10}"); a.axis("off")
        for _ in range(10): B = life_step(B)
    plt.show()

## 2. Hand-wire a CNN that computes Life exactly
Network: `Conv2d(1→4, 3×3, circular padding)` → ReLU → `Conv2d(4→1, 1×1)`. Trick: let the 3×3 kernel compute $s = 2\cdot(\text{neighbours}) + \text{self}$. The cell lives next step **iff $5 \le s \le 7$** (check: alive with 2 → 5, dead with 3 → 6, alive with 3 → 7; dead with 2 → 4 and alive with 4 → 9 die). A "bump" made of ReLUs is exactly 1 on $\{5,6,7\}$ and 0 on every other integer:
$$\text{ReLU}(s-4)-\text{ReLU}(s-5)-\text{ReLU}(s-7)+\text{ReLU}(s-8)$$
Set the weights and biases of `net` accordingly (all 4 channels share the kernel; the biases do the shifting).

In [ ]:
def make_net(channels=4):
    return nn.Sequential(nn.Conv2d(1, channels, 3, padding=1, padding_mode="circular"), nn.ReLU(), nn.Conv2d(channels, 1, 1))

def hand_wire(net):
    # TODO 2: kernel with centre 1 and neighbours 2, biases −4, −5, −7, −8; 1×1 conv weights +1, −1, −1, +1
    raise NotImplementedError  # TODO 2

In [ ]:
boards = torch.tensor((np.random.default_rng(1).random((500, 1, 32, 32)) < 0.35), dtype=torch.float32)
def _reference_step(bd):   # a reference implementation, so this cell works before TODO 1 is done
    n = sum(np.roll(np.roll(bd, dy, 0), dx, 1) for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0))
    return ((n == 3) | ((bd == 1) & (n == 2))).astype(int)
targets = torch.tensor(np.stack([_reference_step(bd[0].numpy().astype(int)) for bd in boards])[:, None], dtype=torch.float32)
def _t_hand():
    net = make_net(); hand_wire(net)
    with torch.no_grad(): out = net(boards)
    return torch.equal(out, targets)
check("hand-wired CNN", _t_hand, "The output must equal the next Life board exactly (not just after rounding) on 500 random boards.")

## 3. Can gradient descent find it?
Same architecture, random initialisation, binary cross-entropy on (board, next board) pairs. We count a run as a success if it gets **every** cell of the test boards right. Try the minimal 4 channels and an over-parameterised 32, three seeds each.

In [ ]:
def train(channels, seed, steps=1500):
    torch.manual_seed(seed); net = make_net(channels); opt = torch.optim.Adam(net.parameters(), 1e-2)
    for step in range(steps):
        idx = torch.randint(0, 400, (32,))
        loss = F.binary_cross_entropy_with_logits(net(boards[idx]), targets[idx]); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): acc = ((net(boards[400:]) > 0).float() == targets[400:]).float().mean().item()
    return acc

if ready("hand-wired CNN"):
    res = {c: [train(c, s) for s in range(3)] for c in (4, 32)}
    for c, accs in res.items():
        print(f"{c:>2} channels: cell accuracy per seed {[f'{a:.4f}' for a in accs]}  → perfect in {sum(a == 1.0 for a in accs)}/3 runs")
    print("→ The 4-channel solution exists (you built it!) but gradient descent doesn't reliably find it from random weights; extra width makes success more likely (the lottery-ticket idea).")

<details><summary>▶ Solution</summary>

```python
def life_step(board):
    n = sum(np.roll(np.roll(board, dy, 0), dx, 1) for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0))
    return ((n == 3) | ((board == 1) & (n == 2))).astype(board.dtype)
```

```python
def hand_wire(net):
    k = torch.full((3, 3), 2.0); k[1, 1] = 1.0              # s = 2·neighbours + self
    with torch.no_grad():
        net[0].weight.copy_(k.expand(4, 1, 3, 3))
        net[0].bias.copy_(torch.tensor([-4.0, -5.0, -7.0, -8.0]))
        net[2].weight.copy_(torch.tensor([1.0, -1.0, -1.0, 1.0]).view(1, 4, 1, 1))
        net[2].bias.zero_()
```
</details>

## Stretch goals
- Train to predict **n steps** ahead with a deeper net (the paper's harder setting). How does the success rate drop?
- Visualise the learned 3×3 kernels of a successful 32-channel run: do some look like your hand-wired one?
- Initialise near your hand-wired solution plus noise: how much noise can SGD recover from?